In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, classification_report, roc_auc_score,
    balanced_accuracy_score
)
from sklearn.model_selection import train_test_split # Import que faltava
import matplotlib.pyplot as plt
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
import seaborn as sns

# Configurações
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"

# Configurações de visualização
%matplotlib inline
sns.set_style('whitegrid')
plt.style.use('fivethirtyeight')
pd.set_option('display.max_columns', None) # Mostrar todas as colunas

print("Bibliotecas importadas e configurações aplicadas.")

Bibliotecas importadas e configurações aplicadas.


In [2]:
# --- 1. Definição de Caminhos ---

# Caminho base onde as pastas de artefatos de modelo (random_forest, xgboost, etc.) estão
model_base_path = '..\\..\\data\\preprocessors-pipeline' 

# Caminho para o dataset de validação gerado
validation_dataset_path = '..\\..\\data\\processed\\ubuntu_server_dataset_final_argus.csv'

print(f"Caminho base dos modelos: {model_base_path}")
print(f"Caminho do dataset de validação: {validation_dataset_path}")

Caminho base dos modelos: ..\..\data\preprocessors-pipeline
Caminho do dataset de validação: ..\..\data\processed\ubuntu_server_dataset_final_argus.csv


In [3]:
# --- 2. Carregamento e Limpeza Pós-Leitura do Dataset de Validação ---

print(f"Carregando dataset de validação: {validation_dataset_path}...")
df_validation = pd.read_csv(validation_dataset_path, low_memory=False)

print("\n--- Verificação e Limpeza de NaNs (Pós-Leitura) ---")
print(f"DataFrame contém NaNs ANTES do tratamento: {df_validation.isnull().values.any()}")

# Coluna específica do problema
coluna_problema = 'response_body_len'

if coluna_problema in df_validation.columns:
    if df_validation[coluna_problema].isnull().values.any():
        print(f"Tratando NaNs encontrados na coluna '{coluna_problema}'...")
        # Preenche NaNs com 0
        df_validation[coluna_problema] = df_validation[coluna_problema].fillna(0)
        
        # Por segurança, também substitui infinitos (se houver)
        df_validation[coluna_problema] = df_validation[coluna_problema].replace([np.inf, -np.inf], 0)
        
        print(f"NaNs e Infs em '{coluna_problema}' preenchidos com 0.")
    else:
        print(f"Coluna '{coluna_problema}' checada, não contém NaNs.")
else:
    print(f"Aviso: Coluna '{coluna_problema}' não encontrada.")

# Verificação final
print(f"DataFrame contém NaNs APÓS o tratamento: {df_validation.isnull().values.any()}")
if df_validation.isnull().values.any():
    print("\nATENÇÃO: NaNs ainda persistem! Colunas com NaNs:")
    print(df_validation.isnull().sum()[df_validation.isnull().sum() > 0])

Carregando dataset de validação: ..\..\data\processed\ubuntu_server_dataset_final_argus.csv...

--- Verificação e Limpeza de NaNs (Pós-Leitura) ---
DataFrame contém NaNs ANTES do tratamento: False
Coluna 'response_body_len' checada, não contém NaNs.
DataFrame contém NaNs APÓS o tratamento: False


In [4]:
# --- 3. Preparação dos Dados para Validação ---

print("Separando features (X_val) e rótulos (y_val)...")

# Define as colunas de rótulo
label_cols = ['attack_cat', 'label']

# X_val: Todas as colunas, exceto os rótulos
X_val = df_validation.drop(columns=label_cols)

# y_val_text: O rótulo categórico (ex: 'Normal', 'DoS', 'Exploits')
y_val_text = df_validation['attack_cat']

# y_val_binary: O rótulo binário (0 para 'Normal', 1 para 'Attack')
# Mapeia 'Normal' -> 0 e qualquer outra coisa -> 1
y_val_binary = df_validation['attack_cat'].apply(lambda x: 0 if x == 'Normal' else 1)

print(f"X_val (features) shape: {X_val.shape}")
print(f"y_val_text (rótulos multiclasse) shape: {y_val_text.shape}")
print(f"y_val_binary (rótulos binários) shape: {y_val_binary.shape}")

print("\nVerificação de nulos nos dados prontos:")
print(f"  X_val contém nulos: {X_val.isnull().values.any()}")
print(f"  y_val_text contém nulos: {y_val_text.isnull().values.any()}")
print(f"  y_val_binary contém nulos: {y_val_binary.isnull().values.any()}")

Separando features (X_val) e rótulos (y_val)...
X_val (features) shape: (340419, 42)
y_val_text (rótulos multiclasse) shape: (340419,)
y_val_binary (rótulos binários) shape: (340419,)

Verificação de nulos nos dados prontos:
  X_val contém nulos: False
  y_val_text contém nulos: False
  y_val_binary contém nulos: False


In [5]:
# --- 4. Modelos a Validar ---

# Dicionário mapeando Nome (Chave) para a pasta de artefatos (Valor)
model_directories = {
    'RandomForest': 'RF',
    'XGBoost': 'XGB',
    'SVM': 'SVM',
    'SelfTraining-RF': 'SelfTraining-RF',
    'SelfTraining-XGB': 'SelfTraining-XGB',
    'IsolationForest': 'IF',
    'IsolationForestSubsampled': 'IF-Subsampled',
    'KMeans': 'KMeans',
    'MBKMeans': 'MBKMeans',
    'HDBSCAN': 'HDBSCAN',
    'DBSCAN': 'DBSCAN',
    'Encoder': 'Encoder'
}

print("Lista de modelos para validação definida.")

Lista de modelos para validação definida.


In [6]:
# --- 5. Função Auxiliar de Carregamento ---

def load_artifact(file_path):
    """
    Carrega um artefato (modelo, pipeline, etc.) de um arquivo .joblib ou .json.
    Retorna None se o arquivo não for encontrado ou se houver um erro.
    """
    if not os.path.exists(file_path):
        # print(f"  Aviso: Artefato não encontrado: {file_path}")
        return None
    try:
        if file_path.endswith('.joblib'):
            return joblib.load(file_path)
        elif file_path.endswith('.json'):
            with open(file_path, 'r') as f:
                return json.load(f)
    except Exception as e:
        print(f"  Erro ao carregar artefato {file_path}: {e}")
        return None

print("Função 'load_artifact' definida.")

Função 'load_artifact' definida.


In [7]:
# --- 6. Definição de Grupos de Modelos e Carregamento do Encoder ---

# Modelos que preveem 'attack_cat' e PRECISAM do LabelEncoder multiclasse
MODELS_MULTICLASS = [
    'RandomForest',
    'XGBoost',
    'SelfTraining-RF',
    'SelfTraining-XGB',
    'SVM'
]

# Modelos que preveem Anomalia (binário) e NÃO usam o encoder multiclasse
MODELS_ANOMALY = [
    'IsolationForest',
    'IsolationForestSubsampled',
    'KMeans',
    'MBKMeans',
    'HDBSCAN',
    'DBSCAN'
]

shared_multiclass_encoder = None
encoder_dir = model_directories.get('Encoder')
print(encoder_dir)
if encoder_dir:
    # Este é o caminho correto baseado em suas informações
    encoder_path = os.path.join(model_base_path, encoder_dir, 'encoder.joblib')
    
    print(f"Tentando carregar o encoder compartilhado de: {encoder_path}")
    shared_multiclass_encoder = load_artifact(encoder_path)

if shared_multiclass_encoder:
    print(">>> Sucesso: Encoder multiclasse compartilhado foi carregado.")
else:
    print(">>> ERRO CRÍTICO: Não foi possível carregar o 'shared_multiclass_encoder'.")
    print(">>> Modelos multiclasse (RF, XGB, etc.) podem falhar.")

Encoder
Tentando carregar o encoder compartilhado de: ..\..\data\preprocessors-pipeline\Encoder\encoder.joblib
>>> Sucesso: Encoder multiclasse compartilhado foi carregado.


In [8]:
# --- 7. Loop Principal de Validação ---

print("\n--- Iniciando Loop de Validação dos Modelos ---")

# Dicionário para armazenar resultados (métricas)
results = {}
# Dicionário para armazenar matrizes de confusão (para plotagem)
confusion_matrices = {}

# Mapeia y_val_text para binário (para modelos não supervisionados)
# 'Normal' -> 0, 'Attack' (qualquer tipo) -> 1
y_val_binary_mapped = y_val_text.apply(lambda x: 0 if x == 'Normal' else 1).values

# Iterar sobre cada modelo definido no dicionário
for model_key, model_dir in model_directories.items():



    # DEBUG: verificar rótulos no validation set
    print("Unique labels in validation (y_val_text):", sorted(y_val_text.unique()))
    print("Count per label in validation:")
    print(y_val_text.value_counts())
    
    # Se o encoder foi carregado, inspecione
    if shared_multiclass_encoder is not None:
        try:
            encoder_classes = getattr(shared_multiclass_encoder, "classes_", None)
            print("Encoder type:", type(shared_multiclass_encoder))
            print("Encoder classes_:", encoder_classes)
        except Exception as e:
            print("Erro ao inspecionar encoder:", e)
    else:
        print("shared_multiclass_encoder is None")


    
    model_name = model_key # 'RandomForest', 'XGBoost', etc.
    print(f"\nValidando: {model_name}...")
    
    try:
        # 1. Definir caminho para os artefatos do modelo atual
        artifacts_path = os.path.join(model_base_path, model_dir)
        if not os.path.isdir(artifacts_path):
            print(f"  Aviso: Diretório de artefatos não encontrado, pulando: {artifacts_path}")
            continue

        # 2. Procurar e carregar artefatos de forma flexível
        pipeline = None
        model = None
        preprocessor = None
        threshold = None
        
        # Listar arquivos da pasta de artefatos (se existir)
        files = []
        try:
            files = os.listdir(artifacts_path)
        except Exception:
            files = []
        
        # Função auxiliar para carregar se existir
        def try_load(fname):
            try:
                return load_artifact(os.path.join(artifacts_path, fname))
            except Exception:
                return None
        
        # Tentar carregamentos "óbvios" (nomes exatos)
        for suffix in (f'pipeline_{model_dir}.joblib', f'model_{model_dir}.joblib', f'preprocessor_{model_dir}.joblib', f'threshold_{model_dir}.json'):
            if suffix in files:
                if suffix.startswith('pipeline'):
                    pipeline = try_load(suffix)
                elif suffix.startswith('model'):
                    model = try_load(suffix)
                elif suffix.startswith('preprocessor'):
                    preprocessor = try_load(suffix)
                elif suffix.endswith('.json'):
                    threshold = try_load(suffix)
        
        # Se não encontrou com nomes exatos, faça uma busca permissiva por palavras-chave
        if pipeline is None or model is None or preprocessor is None or threshold is None:
            for f in files:
                lf = f.lower()
                if pipeline is None and 'pipeline' in lf and f.lower().endswith(('.joblib', '.pkl')):
                    pipeline = try_load(f)
                if model is None and ('model' in lf or 'clf' in lf or 'estimator' in lf) and f.lower().endswith(('.joblib', '.pkl')):
                    model = try_load(f)
                if preprocessor is None and 'preprocessor' in lf and f.lower().endswith(('.joblib', '.pkl')):
                    preprocessor = try_load(f)
                if threshold is None and f.lower().endswith('.json') and ('threshold' in lf or 'params' in lf):
                    threshold = try_load(f)
        
        # DEBUG menor
        print(f"  Artefatos carregados -> pipeline: {pipeline is not None}, model: {model is not None}, preprocessor: {preprocessor is not None}, threshold: {threshold is not None}")
        
        # 3. Gerar Predições (lógica mais permissiva)
        y_pred_encoded = None
        y_pred_text = None
        
        # Se existir um pipeline completo, usá-lo (pipeline já inclui preprocessor + estimator)
        if pipeline is not None:
            try:
                y_pred_encoded = pipeline.predict(X_val)
            except Exception as e:
                print("  Erro ao usar pipeline.predict:", e)
                pipeline = None  # forçar fallback
        
        # Se não houver pipeline, mas houver modelo (pode precisar do preprocessor)
        if y_pred_encoded is None and model is not None:
            # Se houver preprocessor, transformar; senão tentar usar X_val cru
            X_in = preprocessor.transform(X_val) if preprocessor is not None else X_val
        
            # Atenção: alguns algoritmos (DBSCAN/HDBSCAN) não implementam predict() para dados novos.
            model_cls_name = model.__class__.__name__.lower()
            if 'dbscan' in model_cls_name or 'hdbscan' in model_cls_name:
                # Ideal: se o modelo foi salvo já ajustado e tiver método de "approximate predict" (hdbscan)
                if hasattr(model, 'predict'):
                    y_pred_encoded = model.predict(X_in)
                elif hasattr(model, 'approximate_predict'):  # hdbscan has approximate_predict (if package used)
                    try:
                        y_pred_encoded, _ = model.approximate_predict(X_in)
                    except Exception as e:
                        print("  Erro no approximate_predict de HDBSCAN:", e)
                        y_pred_encoded = None
                else:
                    # Não existe um método de predição para novos pontos → não faz sentido chamar fit() aqui (seria re-treinar)
                    print(f"  Aviso: {model.__class__.__name__} não suporta inferência direta em novos dados sem re-treinamento. Pulando '{model_name}'.")
                    y_pred_encoded = None
            else:
                # Modelos habituais com predict
                try:
                    y_pred_encoded = model.predict(X_in)
                except Exception as e:
                    print(f"  Erro ao chamar model.predict para {model_name}:", e)
                    y_pred_encoded = None
        
        # Caso não existam pipeline nem model, mas exista apenas preprocessor:
        if y_pred_encoded is None and preprocessor is not None:
            # Caso típico: temos preprocessor e talvez o modelo separado.
            # Verificar se já aplicamos a transformação antes.
            if X_val.shape[1] == preprocessor.transformers_[0][2].__len__():
                # X_val ainda é cru, pode transformar
                X_in = preprocessor.transform(X_val)
            else:
                # X_val já está transformado (OneHotEncoder aplicado anteriormente)
                print(f"  Aviso: X_val já parece transformado ({X_val.shape[1]} features). Pulando transformação.")
                X_in = X_val
        
            # Agora verificar se há modelo
            if model is not None:
                try:
                    y_pred_encoded = model.predict(X_in)
                except Exception as e:
                    print("  Erro ao usar model.predict:", e)
                    y_pred_encoded = None
            else:
                print(f"  Apenas preprocessor encontrado para {model_name} — sem modelo para inferência. Pulando.")
                continue
        
        # Se ainda não houve predição válida, pular
        if y_pred_encoded is None:
            print(f"  Aviso: Não foi possível obter predições para '{model_name}'. Pulando.")
            continue

        # 4. Decodificar/Mapear Predições para Texto ('Normal', 'Attack', 'DoS', etc.)
        
        if model_name in MODELS_MULTICLASS:
            print("  Tipo: Classificador Multiclasse. Usando 'shared_multiclass_encoder'...")
        
            # Se o modelo já retorna strings, não inverter novamente
            if isinstance(y_pred_encoded[0], str):
                y_pred_text = y_pred_encoded
            else:
                # Garantir que o encoder existe e aplicar inverse_transform
                y_pred_text = shared_multiclass_encoder.inverse_transform(y_pred_encoded)

        elif model_name in MODELS_ANOMALY:
            # Estes modelos preveem anomalia (binário)
            print("  Tipo: Detector de Anomalia. Mapeando predição binária...")
            
            # Lógica de mapeamento (ajuste conforme necessário)
            if 'IsolationForest' in model_name:
                # IF: 1 é 'Normal', -1 é 'Attack'
                y_pred_text = np.where(y_pred_encoded == 1, 'Normal', 'Attack')
            elif 'KMeans' in model_name or 'MBKMeans' in model_name:
                # KMeans/MBK: 0 é 'Normal' (abaixo do threshold), 1 é 'Attack' (acima)
                y_pred_text = np.where(y_pred_encoded == 0, 'Normal', 'Attack')
            elif 'HDBSCAN' in model_name or 'DBSCAN' in model_name:
                # DBSCAN/HDBSCAN: -1 é 'Attack' (outlier), 0+ é 'Normal' (cluster)
                y_pred_text = np.where(y_pred_encoded == -1, 'Attack', 'Normal')
            else:
                # Fallback genérico (pode estar errado, mas é um chute)
                y_pred_text = np.where(y_pred_encoded == 0, 'Normal', 'Attack')

        else:
            print(f"  Aviso: Modelo {model_name} não classificado (MULTICLASS ou ANOMALY). Pulando.")
            continue
            

        # 5. Calcular Métricas
        print(f"Calculando métricas para: {model_name}")
        
        # Usamos o y_val_text original como 'true'
        # e o y_pred_text decodificado como 'pred'
        
        # Métricas multiclasse
        acc = accuracy_score(y_val_text, y_pred_text)
        bal_acc = balanced_accuracy_score(y_val_text, y_pred_text)
        
        # Métricas binárias (mapeando "Attack" vs "Normal")
        y_true_binary = (y_val_text != 'Normal').astype(int)
        y_pred_binary = (y_pred_text != 'Normal').astype(int)
        
        f1_bin = f1_score(y_true_binary, y_pred_binary)
        recall_bin = recall_score(y_true_binary, y_pred_binary)
        precision_bin = precision_score(y_true_binary, y_pred_binary)
        
        results[model_name] = {
            'Accuracy': acc,
            'Balanced Accuracy': bal_acc,
            'F1 Score (Binary)': f1_bin,
            'Recall (Binary)': recall_bin,
            'Precision (Binary)': precision_bin
        }
        
        # Gerar Matriz de Confusão
        # Obter todos os rótulos únicos de ambos os sets (true e pred)
        labels = sorted(list(set(y_val_text) | set(y_pred_text)))
        cm = confusion_matrix(y_val_text, y_pred_text, labels=labels)
        confusion_matrices[model_name] = (cm, labels)
        
    except Exception as e:
        print(f"  ERRO ao processar {model_name}. Pulando. Detalhe: {e}")
        # import traceback
        # traceback.print_exc() # Descomente para debug completo

print("\n--- Validação de todos os modelos concluída! ---")


--- Iniciando Loop de Validação dos Modelos ---
Unique labels in validation (y_val_text): ['Analysis', 'DoS', 'Exploits', 'Fuzzers', 'Normal', 'Reconnaissance']
Count per label in validation:
attack_cat
Exploits          278010
DoS                44069
Fuzzers            16863
Normal              1190
Reconnaissance       171
Analysis             116
Name: count, dtype: int64
Encoder type: <class 'sklearn.preprocessing._label.LabelEncoder'>
Encoder classes_: ['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']

Validando: RandomForest...
  Erro ao carregar artefato ..\..\data\preprocessors-pipeline\RF\pipeline_RF.joblib: 118
  Erro ao carregar artefato ..\..\data\preprocessors-pipeline\RF\pipeline_RF.joblib: 118
  Artefatos carregados -> pipeline: False, model: False, preprocessor: True, threshold: False
  Aviso: X_val já parece transformado (42 features). Pulando transformação.
  Apenas preprocessor encontrado para RandomForest —

In [9]:
# --- 8. Exibição dos Resultados ---

print("\n--- Tabela de Resultados Comparativos ---")

if results:
    df_results = pd.DataFrame(results).T
    df_results = df_results.sort_values(by='F1 Score (Binary)', ascending=False)
    
    # Formatação para melhor visualização
    styled_results = df_results.style.format({
        'Accuracy': '{:.2%}',
        'Balanced Accuracy': '{:.2%}',
        'F1 Score (Binary)': '{:.4f}',
        'Recall (Binary)': '{:.4f}',
        'Precision (Binary)': '{:.4f}'
    }).background_gradient(
        cmap='viridis', subset=['F1 Score (Binary)', 'Recall (Binary)', 'Balanced Accuracy']
    )
    
    display(styled_results)
    
    # --- 9. Plotagem das Matrizes de Confusão ---
    print("\n--- Matrizes de Confusão (Interativas) ---")
    
    for model_name, (cm, labels) in confusion_matrices.items():
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
                    xticklabels=labels, yticklabels=labels)
        plt.title(f'Matriz de Confusão: {model_name}')
        plt.xlabel('Predição')
        plt.ylabel('Verdadeiro')
        plt.tight_layout()
        filename = f'confusion_matrix_{model_name}.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved: {filename}")

else:
    print("Nenhum resultado foi gerado. Verifique os erros no loop de validação.")


--- Tabela de Resultados Comparativos ---


,Accuracy,Balanced Accuracy,F1 Score (Binary),Recall (Binary),Precision (Binary)
IsolationForest,0.00%,0.00%,0.9982,1.0000,0.9965
MBKMeans,0.00%,0.00%,0.9982,1.0000,0.9965
SVM,0.00%,1.74%,0.9982,1.0000,0.9965
IsolationForestSubsampled,0.01%,0.62%,0.9978,0.9989,0.9966
SelfTraining-XGB,9.20%,27.20%,0.1667,0.0910,0.9852
XGBoost,0.11%,5.60%,0.0034,0.0017,0.4157
KMeans,0.35%,16.67%,0.0000,0.0000,0.0000



--- Matrizes de Confusão (Interativas) ---
Saved: confusion_matrix_XGBoost.png
Saved: confusion_matrix_SVM.png
Saved: confusion_matrix_SelfTraining-XGB.png
Saved: confusion_matrix_IsolationForest.png
Saved: confusion_matrix_IsolationForestSubsampled.png
Saved: confusion_matrix_KMeans.png
Saved: confusion_matrix_MBKMeans.png
